# 环境与数据挂载（工程基础）

In [1]:
# Step 0: 环境准备与 Google Drive 挂载
import os
import sys

# 忽略 Hugging Face Token 警告（本地文件读取无需登录）
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # 避免多线程警告

from google.colab import drive
drive.mount('/content/drive')

# 安装核心依赖
!pip install -q faiss-cpu sentence-transformers jsonlines tqdm pandas numpy
# 注意：faiss-gpu 仅在 GPU 运行时有效，如果切回 CPU 请改为 faiss-cpu

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.3 MB/s eta 0:00:00


# 通用函数（包含缓存管理）

In [4]:
import json
import jsonlines
import faiss
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

# 定义路径（根据你实际 Drive 路径修改）
BASE_PATH = "/content/drive/MyDrive/COMP5201/project/data"
PROCESSED_PATH = f"{BASE_PATH}/processed"
RAW_PATH = f"{BASE_PATH}/open_ragbench/pdf/arxiv"
EMBEDDING_CACHE_DIR = "/content/drive/MyDrive/COMP5201/project/embeddings_cache"
os.makedirs(EMBEDDING_CACHE_DIR, exist_ok=True)

# ==========================================
# 通用数据加载函数
# ==========================================
def load_jsonl(file_path):
    data = []
    with jsonlines.open(file_path, 'r') as reader:
        for obj in tqdm(reader, desc=f"Loading {os.path.basename(file_path)}"):
            data.append(obj)
    return data

def load_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def parse_qrels(qrels_obj):
    """
    统一返回 {qid: [{"doc_id": ..., "section_id": ...}]} 格式。
    """
    qrels_dict = {}
    for qid, val in qrels_obj.items():
        entries = []
        if isinstance(val, str):
            entries.append({"doc_id": val, "section_id": ""})
        elif isinstance(val, dict):
            entries.append({
                "doc_id": str(val.get('doc_id', val.get('id', ''))),
                "section_id": str(val.get('section_id', ''))
            })
        elif isinstance(val, list):
            for item in val:
                if isinstance(item, str):
                    entries.append({"doc_id": item, "section_id": ""})
                elif isinstance(item, dict):
                    entries.append({
                        "doc_id": str(item.get('doc_id', item.get('id', ''))),
                        "section_id": str(item.get('section_id', ''))
                    })
        if entries:
            qrels_dict[str(qid)] = entries
    return qrels_dict

def extract_query_text(query_val):
    if isinstance(query_val, dict):
        return query_val.get('query') or query_val.get('text') or query_val.get('question') or str(query_val)
    return str(query_val)

def clean_arxiv_id(doc_id):
    return doc_id.split('v')[0] if 'v' in doc_id else doc_id

def is_chunk_hit(retrieved_chunk, gt_entries):
    """判断 chunk 是否命中 ground truth"""
    ret_doc_clean = clean_arxiv_id(retrieved_chunk['doc_id'])
    ret_sec = str(retrieved_chunk.get('section_id', ''))
    for gt in gt_entries:
        gt_doc_clean = clean_arxiv_id(gt['doc_id'])
        gt_sec = str(gt['section_id'])
        if ret_doc_clean == gt_doc_clean and ret_sec == gt_sec:
            return True
    return False

# ==========================================
# Embedding 缓存函数
# ==========================================
def get_embedding_cache_path(chunk_file_path, model_name):
    """生成缓存文件路径"""
    chunk_name = os.path.basename(chunk_file_path).replace('.jsonl', '')
    model_safe_name = model_name.replace('/', '_')
    return os.path.join(EMBEDDING_CACHE_DIR, f"{chunk_name}_{model_safe_name}_fp16.npy")

def load_or_compute_embeddings(chunk_texts, model, model_name, chunk_file_path, batch_size=128):
    """如果缓存存在则加载，否则计算并保存为 float16"""
    cache_path = get_embedding_cache_path(chunk_file_path, model_name)
    if os.path.exists(cache_path):
        print(f"✅ 从缓存加载 Embedding: {os.path.basename(cache_path)}")
        embeddings = np.load(cache_path).astype(np.float32)
        if len(embeddings) == len(chunk_texts):
            return embeddings
        else:
            print("⚠️ 缓存长度不匹配，重新计算...")

    print("🔄 计算 Embedding（无缓存）...")
    embeddings = model.encode(
        chunk_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    np.save(cache_path, embeddings.astype(np.float16))
    print(f"💾 Embedding 已缓存至: {cache_path}")
    return embeddings.astype(np.float32)

print("✅ 环境准备完成，缓存目录已创建")

✅ 环境准备完成，缓存目录已创建


# 测试模块 1：Embedding Model 对比（Chunk 级命中，支持缓存）

In [5]:
# ==========================================
# 测试模块 1: Embedding Model 对比（含缓存）
# ==========================================
import gc
import torch

# 实验参数
CHUNK_FILE = f"{PROCESSED_PATH}/chunks_section_aware.jsonl"
EMBED_MODELS = ["all-MiniLM-L6-v2", "BAAI/bge-base-en-v1.5"]
TOP_K = 10
DEVICE = "cuda"
BATCH_SIZE = 128
USE_HALF_PRECISION = True
TEST_QUERY_LIMIT = 100

print("\n" + "="*60)
print("开始测试模块 1: Embedding Model 对比 (Chunk 级命中)")
print("="*60)

# 加载数据
chunks = load_jsonl(CHUNK_FILE)
queries_raw = load_json(f"{RAW_PATH}/queries.json")
qrels_raw = load_json(f"{RAW_PATH}/qrels.json")

chunk_dicts = chunks
chunk_texts = [c["text"] for c in chunks]

query_ids = list(queries_raw.keys())[:TEST_QUERY_LIMIT]
query_texts = [extract_query_text(queries_raw[qid]) for qid in query_ids]

qrels_dict = parse_qrels(qrels_raw)

# 过滤有效 query
valid_query_ids = []
valid_query_texts = []
for qid, qtext in zip(query_ids, query_texts):
    gt_entries = qrels_dict.get(str(qid), [])
    has_valid = False
    for gt in gt_entries:
        gt_doc = gt.get('doc_id', '')
        gt_sec = str(gt.get('section_id', ''))
        if not gt_doc:
            continue
        for c in chunk_dicts:
            c_doc = c.get('doc_id', '')
            c_sec = str(c.get('section_id', ''))
            if clean_arxiv_id(c_doc) == clean_arxiv_id(gt_doc) and c_sec == gt_sec:
                has_valid = True
                break
        if has_valid:
            break
    if has_valid:
        valid_query_ids.append(qid)
        valid_query_texts.append(qtext)

print(f"语料库 chunks: {len(chunk_texts)}")
print(f"原始测试 queries: {len(query_ids)}")
print(f"有效 queries: {len(valid_query_ids)}")

if len(valid_query_ids) == 0:
    print("❌ 没有有效 query")
    sys.exit(1)

results = {}

for model_name in EMBED_MODELS:
    print(f"\n{'='*20} 测试模型: {model_name} {'='*20}")

    model = SentenceTransformer(model_name, device=DEVICE)
    if USE_HALF_PRECISION:
        model.half()
        print("  ℹ️ 已启用半精度 (FP16)")

    # 使用缓存获取 Embedding
    chunk_embeddings = load_or_compute_embeddings(
        chunk_texts, model, model_name, CHUNK_FILE, batch_size=BATCH_SIZE
    )

    dim = chunk_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(chunk_embeddings)

    print("正在检索并计算 Chunk 级 Hit Rate...")
    hit_count = 0
    for qid, q_text in tqdm(zip(valid_query_ids, valid_query_texts), total=len(valid_query_ids), desc=f"Evaluating {model_name}"):
        q_emb = model.encode([q_text], normalize_embeddings=True)
        _, indices = index.search(q_emb, TOP_K)
        retrieved_chunks = [chunk_dicts[idx] for idx in indices[0]]
        gt_entries = qrels_dict.get(str(qid), [])
        if any(is_chunk_hit(rc, gt_entries) for rc in retrieved_chunks):
            hit_count += 1

    hit_rate = hit_count / len(valid_query_ids)
    results[model_name] = hit_rate
    print(f"✅ {model_name} Chunk Hit Rate@{TOP_K}: {hit_rate:.4f}")

    del model, chunk_embeddings, index
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*50)
print("📊 测试模块 1 最终结果汇总 (Chunk 级命中):")
for model, hr in results.items():
    print(f"   {model}: {hr:.4f}")


开始测试模块 1: Embedding Model 对比 (Chunk 级命中)


Loading chunks_section_aware.jsonl: 99773it [00:01, 81386.46it/s]


语料库 chunks: 99773
原始测试 queries: 100
有效 queries: 100

==================== 测试模型: all-MiniLM-L6-v2 ====================


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

  ℹ️ 已启用半精度 (FP16)
🔄 计算 Embedding（无缓存）...


Batches:   0%|          | 0/780 [00:00<?, ?it/s]

💾 Embedding 已缓存至: /content/drive/MyDrive/COMP5201/project/embeddings_cache/chunks_section_aware_BAAI_bge-base-en-v1.5_fp16.npy
正在检索并计算 Chunk 级 Hit Rate...


Evaluating BAAI/bge-base-en-v1.5: 100%|██████████| 100/100 [00:03<00:00, 25.73it/s]


✅ BAAI/bge-base-en-v1.5 Chunk Hit Rate@10: 0.8600

📊 测试模块 1 最终结果汇总 (Chunk 级命中):
   all-MiniLM-L6-v2: 0.8400
   BAAI/bge-base-en-v1.5: 0.8600


# 测试模块 2：Reranker 消融实验（含缓存）

In [23]:
# ==========================================
# 测试模块 2: Reranker 消融实验（含缓存）
# ==========================================
import gc
import torch

CHUNK_FILE = f"{PROCESSED_PATH}/chunks_section_aware.jsonl"
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"
TOP_K_RETRIEVAL = 20
TOP_K_FINAL = 15
DEVICE = "cuda"
BATCH_SIZE = 128
USE_HALF_PRECISION = True
TEST_QUERY_LIMIT = 100

print("\n" + "="*60)
print("开始测试模块 2: Reranker 消融实验 (Chunk 级命中)")
print("="*60)

chunks = load_jsonl(CHUNK_FILE)
queries_raw = load_json(f"{RAW_PATH}/queries.json")
qrels_raw = load_json(f"{RAW_PATH}/qrels.json")

chunk_dicts = chunks
chunk_texts = [c["text"] for c in chunks]

query_ids = list(queries_raw.keys())[:TEST_QUERY_LIMIT]
query_texts = [extract_query_text(queries_raw[qid]) for qid in query_ids]

qrels_dict = parse_qrels(qrels_raw)

# 过滤有效 query
valid_qids = []
valid_qtexts = []
for qid, qtext in zip(query_ids, query_texts):
    gt_entries = qrels_dict.get(str(qid), [])
    has_valid = False
    for gt in gt_entries:
        gt_doc = gt.get('doc_id', '')
        gt_sec = str(gt.get('section_id', ''))
        if not gt_doc:
            continue
        for c in chunk_dicts:
            c_doc = c.get('doc_id', '')
            c_sec = str(c.get('section_id', ''))
            if clean_arxiv_id(c_doc) == clean_arxiv_id(gt_doc) and c_sec == gt_sec:
                has_valid = True
                break
        if has_valid:
            break
    if has_valid:
        valid_qids.append(qid)
        valid_qtexts.append(qtext)

print(f"有效 queries: {len(valid_qids)}")
if len(valid_qids) == 0:
    print("❌ 无有效 query")
    sys.exit(1)

# Embedding 模型
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
if USE_HALF_PRECISION:
    embed_model.half()

# 使用缓存获取 Embedding
chunk_embeddings = load_or_compute_embeddings(
    chunk_texts, embed_model, EMBED_MODEL_NAME, CHUNK_FILE, batch_size=BATCH_SIZE
)

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

# Reranker
reranker = CrossEncoder(RERANK_MODEL_NAME, device=DEVICE)

results = {"no_rerank": {"hit": 0, "latencies": []}, "with_rerank": {"hit": 0, "latencies": []}}

for qid, q_text in tqdm(zip(valid_qids, valid_qtexts), total=len(valid_qids), desc="Processing"):
    gt_entries = qrels_dict.get(str(qid), [])
    q_emb = embed_model.encode([q_text], normalize_embeddings=True)

    # 无 Rerank
    start = time.time()
    _, indices_a = index.search(q_emb, TOP_K_FINAL)
    lat_a = time.time() - start
    results["no_rerank"]["latencies"].append(lat_a)
    ret_chunks_a = [chunk_dicts[i] for i in indices_a[0]]
    if any(is_chunk_hit(rc, gt_entries) for rc in ret_chunks_a):
        results["no_rerank"]["hit"] += 1

    # 有 Rerank
    start = time.time()
    _, indices_20 = index.search(q_emb, TOP_K_RETRIEVAL)
    candidate_chunks = [chunk_dicts[i] for i in indices_20[0]]
    candidate_texts = [c["text"] for c in candidate_chunks]
    pairs = [[q_text, txt] for txt in candidate_texts]
    scores = reranker.predict(pairs)
    sorted_indices = np.argsort(scores)[::-1][:TOP_K_FINAL]
    lat_b = time.time() - start
    results["with_rerank"]["latencies"].append(lat_b)
    final_chunks = [candidate_chunks[i] for i in sorted_indices]
    if any(is_chunk_hit(rc, gt_entries) for rc in final_chunks):
        results["with_rerank"]["hit"] += 1

n = len(valid_qids)
hr_no = results["no_rerank"]["hit"] / n
hr_with = results["with_rerank"]["hit"] / n
avg_lat_no = np.mean(results["no_rerank"]["latencies"]) * 1000
avg_lat_with = np.mean(results["with_rerank"]["latencies"]) * 1000

print("\n" + "="*50)
print("📊 测试模块 2 结果 (Chunk 级命中):")
print(f"无 Rerank: Chunk Hit@{TOP_K_FINAL} = {hr_no:.4f}, 延迟 = {avg_lat_no:.2f} ms")
print(f"有 Rerank: Chunk Hit@{TOP_K_FINAL} = {hr_with:.4f}, 延迟 = {avg_lat_with:.2f} ms")
print(f"提升: {hr_with - hr_no:.4f}")

del embed_model, reranker, chunk_embeddings, index
gc.collect()
torch.cuda.empty_cache()


开始测试模块 2: Reranker 消融实验 (Chunk 级命中)


Loading chunks_section_aware.jsonl: 99773it [00:01, 66051.11it/s]


有效 queries: 100


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 从缓存加载 Embedding: chunks_section_aware_BAAI_bge-base-en-v1.5_fp16.npy


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Processing: 100%|██████████| 100/100 [00:59<00:00,  1.67it/s]



📊 测试模块 2 结果 (Chunk 级命中):
无 Rerank: Chunk Hit@15 = 0.9100, 延迟 = 27.10 ms
有 Rerank: Chunk Hit@15 = 0.9500, 延迟 = 554.45 ms
提升: 0.0400


测试情况：top_K = 3， 5， 10， 15

1.   当问题为20题时：模型命中率仅在top_K = 15 时有0.05提升，其余情况均为负提升或无提升，延迟约为450ms。此情况下不建议Rerank
2.   当问题为50题时：模型命中率在四种情况下均有些许提升，在0.05上下浮动。延迟约为530ms
3.   当问题为100题时：模型命中率在四种情况下均有些许提升，在0.05上下浮动。延迟为550ms

评价：表现提升少，延迟增加大。不建议使用Rerank



# 测试模块 3：全量 Chunking + Embedding 交叉实验（含缓存）

In [11]:
# ==========================================
# 测试模块 3: 全量对比矩阵（含缓存）
# ==========================================
import gc
import torch

CHUNK_FILES = {
    "Fixed": f"{PROCESSED_PATH}/chunks_fixed.jsonl",
    "Recursive": f"{PROCESSED_PATH}/chunks_recursive.jsonl",
    "Section-Aware": f"{PROCESSED_PATH}/chunks_section_aware.jsonl"
}
EMBED_MODELS = ["all-MiniLM-L6-v2", "BAAI/bge-base-en-v1.5"]
TOP_K = 15
DEVICE = "cuda"
BATCH_SIZE = 128
USE_HALF_PRECISION = True
TEST_QUERY_LIMIT = 200

print("\n" + "="*60)
print("开始测试模块 3: 全量 Chunking 策略对比矩阵 (Chunk 级命中)")
print("="*60)

queries_raw = load_json(f"{RAW_PATH}/queries.json")
qrels_raw = load_json(f"{RAW_PATH}/qrels.json")
qrels_dict = parse_qrels(qrels_raw)

query_ids_all = list(queries_raw.keys())[:TEST_QUERY_LIMIT]
query_texts_all = [extract_query_text(queries_raw[qid]) for qid in query_ids_all]

results_table = []

for strategy_name, chunk_path in CHUNK_FILES.items():
    print(f"\n{'#'*50}\n处理 Chunking: {strategy_name}\n{'#'*50}")

    chunks = load_jsonl(chunk_path)
    chunk_dicts = chunks
    chunk_texts = [c["text"] for c in chunks]

    # 过滤有效 query
    valid_qids = []
    valid_qtexts = []
    for qid, qtext in zip(query_ids_all, query_texts_all):
        gt_entries = qrels_dict.get(str(qid), [])
        has_valid = False
        for gt in gt_entries:
            gt_doc = gt.get('doc_id', '')
            gt_sec = str(gt.get('section_id', ''))
            if not gt_doc:
                continue
            for c in chunk_dicts:
                c_doc = c.get('doc_id', '')
                c_sec = str(c.get('section_id', ''))
                if clean_arxiv_id(c_doc) == clean_arxiv_id(gt_doc) and c_sec == gt_sec:
                    has_valid = True
                    break
            if has_valid:
                break
        if has_valid:
            valid_qids.append(qid)
            valid_qtexts.append(qtext)

    print(f"  有效 queries: {len(valid_qids)}")

    if len(valid_qids) == 0:
        print("  ⚠️ 跳过该策略（无有效 query）")
        for model_name in EMBED_MODELS:
            results_table.append({"Chunking": strategy_name, "Embedding": model_name, f"ChunkHit@{TOP_K}": None})
        continue

    for model_name in EMBED_MODELS:
        print(f"\n  --- 模型: {model_name} ---")
        model = SentenceTransformer(model_name, device=DEVICE)
        if USE_HALF_PRECISION:
            model.half()

        # 使用缓存获取 Embedding
        chunk_embeddings = load_or_compute_embeddings(
            chunk_texts, model, model_name, chunk_path, batch_size=BATCH_SIZE
        )

        index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
        index.add(chunk_embeddings)

        hit_count = 0
        for qid, q_text in tqdm(zip(valid_qids, valid_qtexts), total=len(valid_qids), desc=f"  Evaluating {model_name}"):
            q_emb = model.encode([q_text], normalize_embeddings=True)
            _, indices = index.search(q_emb, TOP_K)
            retrieved_chunks = [chunk_dicts[i] for i in indices[0]]
            gt_entries = qrels_dict.get(str(qid), [])
            if any(is_chunk_hit(rc, gt_entries) for rc in retrieved_chunks):
                hit_count += 1

        hit_rate = hit_count / len(valid_qids)
        results_table.append({"Chunking": strategy_name, "Embedding": model_name, f"ChunkHit@{TOP_K}": hit_rate})
        print(f"  ✅ ChunkHit@{TOP_K} = {hit_rate:.4f}")

        del model, chunk_embeddings, index
        gc.collect()
        torch.cuda.empty_cache()

df = pd.DataFrame(results_table)
print("\n" + "="*60)
print("📋 最终实验结果表 (Chunk 级命中):")
print(df.to_string(index=False))

output_path = "/content/drive/MyDrive/DSAI5201/results/top_K_15.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"\n💾 结果已保存至: {output_path}")


开始测试模块 3: 全量 Chunking 策略对比矩阵 (Chunk 级命中)

##################################################
处理 Chunking: Fixed
##################################################


Loading chunks_fixed.jsonl: 87769it [00:01, 61708.64it/s]


  有效 queries: 200

  --- 模型: all-MiniLM-L6-v2 ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 从缓存加载 Embedding: chunks_section_aware_BAAI_bge-base-en-v1.5_fp16.npy


  Evaluating BAAI/bge-base-en-v1.5: 100%|██████████| 200/200 [00:07<00:00, 25.06it/s]


  ✅ ChunkHit@15 = 0.9200

📋 最终实验结果表 (Chunk 级命中):
     Chunking             Embedding  ChunkHit@15
        Fixed      all-MiniLM-L6-v2        0.890
        Fixed BAAI/bge-base-en-v1.5        0.905
    Recursive      all-MiniLM-L6-v2        0.895
    Recursive BAAI/bge-base-en-v1.5        0.915
Section-Aware      all-MiniLM-L6-v2        0.900
Section-Aware BAAI/bge-base-en-v1.5        0.920

💾 结果已保存至: /content/drive/MyDrive/DSAI5201/results/top_K_15.csv
